# Sóstrato el Arquitecto · MODELOS

Proyecta la ocupación de salas y laboratorios y genera la **programación sintética explícita**. Lee lo que deja `PIPELINES.ipynb` en `BBDD/` y escribe los insumos del deploy.

**Modelo adoptado (D14 + D15):**
- **Mapa:** **C1** = w · naive + (1 − w) · G1b en 2027; desde 2028, **C1E** = w · naive escalado + (1 − w) · G1b (D19, 28-10). La fuente de volumen y w se leen del banco (§4e, D15): se usa la variante con volumen **proyectado** que pasa la regla de adopción (hoy **C1-TEN** en pregrado, con tendencia por campus); en AV, cuyos orígenes son informativos, la de menor MASE (hoy **C1-EUR**). Si ninguna pasara, el naive.
  - **Naive:** el mapa real del último período del mismo sufijo.
  - **Naive escalado (D19, 2028–2031):** el naive multiplicado en cada campus × clase por el crecimiento de salas que implica el objetivo de G1b (Σ volumen objetivo con β ÷ Σ volumen base). Sin escalar, el naive es una copia que no crece: con w = 0,8 el centro del mapa crecía solo un 4 % entre 2027 y 2031 contra un 24 % de los inscritos.
  - **G1b:** las secciones físicas reales de ese período, remuestreadas por estrato hasta el volumen proyectado, con elasticidad salas/inscritos β.
- **Número de secciones** de la programación explícita: el **estándar real** de Diego (H2). Por sede × componente es el percentil del tamaño de sección que reproduce lo que se programó en el último período real; secciones = inscritos ÷ estándar.
- **Programación explícita:** las secciones del último período real se remuestrean hasta ese número y se ubican con la reasignación a sala libre equivalente. Lo que no cabe es déficit.
- **Rango de capacidad (D18, 27-10):** cada sección necesita un rango de sala según sus inscritos proyectados (0-20, 21-40, 41-60, 60 o más, los de la planta). La alerta es **anidada hacia arriba**: una sección chica puede usar una sala grande, pero no al revés. La programación explícita solo deja una sección en una sala si cabe.

**Primera versión (decisiones del 26-10):**
- **2027** (próximo semestre de cada sufijo): crecimiento de la **variante adoptada** en el banco; **Euribia** aporta la banda bajo/alto (D15: C1-EUR no le gana al naive en pregrado).
- **2028–2031** con el crecimiento de **Ariadna**. Desde el 27-10 se apoya en el pseudo-backtest de largo plazo del banco (§7, D17): se usa C1 en todos los años (decisión de Diego del 27-10: la regla por horizonte alternaba C1 y naive por diferencias de ±0,02), la banda se ensancha con k_h y los escenarios bajo/alto de Ariadna se ensanchan si su propio backtest lo pide. 2028–2030 (H+2 … H+4) quedan **validados con muestra chica**; 2031 (H+5) queda **extrapolado**.
- Las funciones definitivas viven aquí. `VALIDACION_WALKFORWARD.ipynb` conserva la copia que se validó, y §8 comprueba que ambas den lo mismo en 202620.

| Sección | Salida en `BBDD/` |
|---|---|
| 0 · Configuración | — |
| 1 · Insumos y controles | — |
| 2 · Funciones del modelo | — |
| 3 · Volumen proyectado | `VOLUMEN_PROYECTADO.parquet` |
| 4 · Número de secciones (estándar real) | `SECCIONES_PROYECTADAS.parquet` |
| 5 · Mapa con C1 | `MAPA_PROYECTADO.parquet` |
| 6 · Programación sintética explícita | `PROGRAMACION_SINTETICA.parquet` |
| 7 · Ocupación contra la oferta | `OCUPACION_Y_DEFICIT.parquet`, `OCUPACION_POR_RANGO.parquet` |
| 8 · Controles y metadatos | `MODELO_METADATOS.json` |

## 0 · Configuración

In [35]:
import json
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name.startswith("Sóstrato el Arquitecto"))
REPO_DP = RAIZ.parent
BBDD = RAIZ / "BBDD"
DESCARGAS = Path.home() / "Downloads"
EURIBIA = DESCARGAS / "Proyección Demanda Asiganturas BBDD"
ARIADNA = DESCARGAS / "MODELO_PROYECCCIÓN_SIMPLE" / "PROYECCION_MATRICULA_2026_2031_28092026.parquet"

SUFIJOS = {10: "UG", 20: "UG", 5: "AV", 15: "AV", 25: "AV"}
ANIO_CORTO = 2027                      # Euribia
ANIOS_LARGO = [2028, 2029, 2030, 2031] # Ariadna (pseudo-backtest del banco §7: H+2 … H+4 medidos, H+5 extrapolado)
ESCENARIOS = ["bajo", "base", "alto"]
N_SIM = {"corto": 100, "largo": 50}
SEMILLA = 2049
CLASES = ["SALA", "LABORATORIO"]
DIAS = ["LUNES", "MARTES", "MIERCOLES", "JUEVES", "VIERNES", "SABADO", "DOMINGO"]
NB = 20
KS_REAL = list(range(5, 100, 5))
COMPONENTES_SIN_HORARIO = {"OCS"}
RAZON_MIN, RAZON_MAX = 0.25, 4.0

# Parámetros validados en el banco: w de C1 en el origen más reciente de cada nivel
val = pd.read_parquet(BBDD / "VALIDACION_RESULTADOS.parquet")
# D15, por nivel. UG: la variante con volumen PROYECTADO que pasa la regla de adopción (mayoría de orígenes que deciden,
# en celda y campus × día × bloque) con menor MASE medio; si ninguna pasa, el naive (persistencia con w = 1).
# AV: sus orígenes son informativos, así que se toma la variante de menor MASE medio en ellos (hoy C1-EUR).
VARIANTES = {"C1-EUR": "EUR", "C1-PER": "PER", "C1-TEN": "TEN"}
GRANOS_DECIDEN = ["celda", "campus × día × bloque"]
v = val[val["MODELO"].isin(VARIANTES) & val["GRANO"].isin(GRANOS_DECIDEN)].assign(NIVEL=lambda d: d["ORIGEN"].mod(100).map(SUFIJOS))
dec = v[v["USO"] == "decide"]
gana = dec.assign(G=dec["MASE"] < 1 - 1e-9).groupby(["MODELO", "GRANO"])["G"].sum().unstack()
pasa = gana[(gana > dec["ORIGEN"].nunique() / 2).all(axis=1)].index
MODELO_ADOPTADO = {"UG": dec[dec["MODELO"].isin(pasa)].groupby("MODELO")["MASE"].mean().idxmin() if len(pasa) else None,
                   "AV": v[v["NIVEL"] == "AV"].groupby("MODELO")["MASE"].mean().idxmin()}
FUENTE_VOLUMEN = {niv: VARIANTES.get(m, "PER") for niv, m in MODELO_ADOPTADO.items()}
W_NAIVE = {}
for niv, m in MODELO_ADOPTADO.items():
    c = val[(val["MODELO"] == m) & (val["ORIGEN"].mod(100).map(SUFIJOS) == niv)].drop_duplicates("ORIGEN").sort_values("ORIGEN")
    W_NAIVE[niv] = float(c["W_NAIVE"].iloc[-1]) if m else 1.0
print("victorias por variante (orígenes que deciden):"); print(gana)
print("MASE medio AV:", v[v["NIVEL"] == "AV"].groupby("MODELO")["MASE"].mean().round(3).to_dict())
print("modelo adoptado:", MODELO_ADOPTADO, "· fuente de volumen:", FUENTE_VOLUMEN, "· w (naive):", W_NAIVE)

# D19 · desde H+2 el naive se escala por el crecimiento de salas de G1b (Diego, 28-10). A 1 año C1E pierde contra C1-TEN
# en el banco, así que 2027 queda con C1; a largo plazo corrige el sesgo (H+4: −7,6 % → −2,7 %) y mejora el MASE desde H+3,
# pero en H+2 lo empeora en 0,011. El w de C1E es el de su variante en el banco.
ESCALAR_DESDE_H = 2
MODELO_LP = {niv: (m.replace("C1-", "C1E-") if m else None) for niv, m in MODELO_ADOPTADO.items()}
W_NAIVE_LP = {}
for niv, m in MODELO_LP.items():
    c = val[(val["MODELO"] == m) & (val["ORIGEN"].mod(100).map(SUFIJOS) == niv)].drop_duplicates("ORIGEN").sort_values("ORIGEN")
    W_NAIVE_LP[niv] = float(c["W_NAIVE"].iloc[-1]) if m and len(c) else W_NAIVE[niv]
print("largo plazo (D19):", MODELO_LP, "· w (naive escalado):", W_NAIVE_LP)

# D17 · largo plazo, por horizonte (banco §7): modelo del año, factor k_h de la banda y ensanche de los escenarios de Ariadna
bh = pd.read_parquet(BBDD / "BANDA_HORIZONTE.parquet")
ae = pd.read_parquet(BBDD / "ARIADNA_ERROR_HORIZONTE.parquet")
H_VALIDADO = int(bh.loc[bh["K_ORIGEN"] == "medido", "H"].max())
# C1 en todo el largo plazo (Diego, 27-10): MODELO_H del banco queda solo como informativo
MODELO_H = bh[bh["GRANO"] == "celda"].drop_duplicates(["NIVEL", "H"]).set_index(["NIVEL", "H"])["MODELO_H"].to_dict()
K_H = bh[(bh["GRANO"] == "celda") & (bh["MODELO_BANDA"] == "C1E")].set_index(["NIVEL", "H"])["K"].to_dict()   # k_h de C1E (D19)
ENSANCHE_ARI = ae.set_index(["NIVEL", "H"])["ENSANCHE"].to_dict()
H_MAX_ARI = int(ae["H"].max())
print("regla por horizonte del banco (informativa, no se aplica):", MODELO_H); print("k_h:", {k: round(v, 2) for k, v in K_H.items()})
print("ensanche de Ariadna:", {k: round(v, 2) for k, v in ENSANCHE_ARI.items()})

victorias por variante (orígenes que deciden):
GRANO   campus × día × bloque  celda
MODELO                              
C1-EUR                      0      0
C1-PER                      1      0
C1-TEN                      3      3
MASE medio AV: {'C1-EUR': 0.976, 'C1-PER': 1.0, 'C1-TEN': 1.0}
modelo adoptado: {'UG': 'C1-TEN', 'AV': 'C1-EUR'} · fuente de volumen: {'UG': 'TEN', 'AV': 'EUR'} · w (naive): {'UG': 0.8, 'AV': 0.0}
largo plazo (D19): {'UG': 'C1E-TEN', 'AV': 'C1E-EUR'} · w (naive escalado): {'UG': 1.0, 'AV': 1.0}
regla por horizonte del banco (informativa, no se aplica): {('UG', 1): 'NAIVE', ('UG', 2): 'C1', ('UG', 3): 'NAIVE', ('UG', 4): 'C1', ('UG', 5): 'C1', ('AV', 1): 'C1', ('AV', 2): 'C1', ('AV', 3): 'C1', ('AV', 4): 'NAIVE', ('AV', 5): 'NAIVE'}
k_h: {('UG', 1): 1.0, ('UG', 2): 1.31, ('UG', 3): 1.44, ('UG', 4): 1.72, ('UG', 5): 1.82, ('AV', 1): 1.0, ('AV', 2): 1.26, ('AV', 3): 1.43, ('AV', 4): 1.67, ('AV', 5): 1.76}
ensanche de Ariadna: {('AV', 1): 1.0, ('AV', 2): 1.44, (

## 1 · Insumos y controles

In [36]:
ph = pd.read_parquet(BBDD / "PROG_HIST_NRC.parquet")
pl = pd.read_parquet(BBDD / "PLANTILLAS_NRC.parquet",
                     columns=["PERIODO", "NRC", "COD_COMPONENTE", "DIA", "BLOQUE", "COD_CAMPUS_EDIFICIO",
                              "EDIFICIO", "SALA", "CLASE_ESPACIO", "PESO_SEMANAS", "PESO_LC"])
pl["CON_SALA"] = pl["CLASE_ESPACIO"] != "SIN SALA"
unidad = pd.read_parquet(BBDD / "UNIDAD_ACADEMICA.parquet")
oferta = pd.read_parquet(BBDD / "OFERTA_ESPACIOS.parquet")
pf = pd.read_csv(REPO_DP / "Heat-Map" / "BBDD" / "PLANTA_FISICA_PREPROCESADA.csv", sep=None, engine="python")
pf = pf[pf["Es_Mas_Actualizado"] == 1].copy()
pf["EDIFICIO"] = pf["Código Edificio"].astype(str); pf["SALA"] = pf["Número Sala"].astype(str)
planta = pf.drop_duplicates(["EDIFICIO", "SALA"]).set_index(["EDIFICIO", "SALA"])["CLASIFICACIÓN"]
moda_edificio = pf.groupby("EDIFICIO")["CLASIFICACIÓN"].agg(lambda s: s.mode().iat[0])
eur = pd.read_csv(EURIBIA / "PROYECCION_INSCRITOS_2026_2027.csv", sep=";", decimal=",", dtype={"CURSO": str, "PERIODO": int}, low_memory=False)
ari = pd.read_parquet(ARIADNA, columns=["periodo", "cod_carrera", "sede", "n", "n_bajo", "n_alto", "tipo"])
ari["periodo"] = ari["periodo"].astype(int)

periodos = set(pl["PERIODO"].unique())
BASE = {s: max(p for p in periodos if p % 100 == s) for s in SUFIJOS}      # último período real de cada sufijo
OBJETIVOS = [(ANIO_CORTO * 100 + s, "corto") for s in SUFIJOS] + [(a * 100 + s, "largo") for a in ANIOS_LARGO for s in SUFIJOS]
horizonte = lambda T: T // 100 - BASE[T % 100] // 100          # años entre la base y el objetivo (2027 = H+1)
validado = lambda T: horizonte(T) <= H_VALIDADO
print("base por sufijo:", BASE)
assert all(BASE[s] // 100 >= ANIO_CORTO - 1 for s in SUFIJOS), "Falta el último año real en algún sufijo"
if 25 in BASE:
    print("aviso: la base AV 202625 viene de una extracción con la inscripción todavía abierta")

base por sufijo: {10: 202610, 20: 202620, 5: 202605, 15: 202615, 25: 202625}
aviso: la base AV 202625 viene de una extracción con la inscripción todavía abierta


## 2 · Funciones del modelo

Son las mismas reglas validadas en el banco:
- **Clasificación de salas** (D10).
- **Sección física** (D11 + corrección del 26-10): se unen las LC y las secciones con horario prácticamente idéntico (≥ 80 % de los casilleros de la más grande).
- **Elasticidad β** (G1b).
- **Estándar real** (H2).
- **Asignación a sala libre equivalente** (D18: con capacidad y tolerancia).
- **Rango de capacidad (D18):** `rango_de` ubica unos inscritos en el rango de sala que necesitan; `tolerancia` mide, por clase, cuánto se pasan las secciones reales de la capacidad de su sala en el período base (percentil 90 de inscritos ÷ capacidad por sala-bloque, piso 1). En 2026 el 6–9 % de las salas-bloque y el ~20 % de los laboratorios-bloque superan su capacidad nominal, así que sin tolerancia el modelo inventaría déficit que hoy no existe.

In [37]:
CAMP = {c: i for i, c in enumerate(sorted(set(pl["COD_CAMPUS_EDIFICIO"].dropna()) | set(ph["COD_CAMPUS"].dropna())))}
DIA_I = {d: i for i, d in enumerate(DIAS)}
CLA_I = {c: i for i, c in enumerate(CLASES)}
N_SC = len(DIAS) * NB * len(CLASES)
N_CELDAS = len(CAMP) * N_SC
UNIDAD_MAT = unidad.set_index("COD_MATERIA")["DESC_UNIDAD_ACADEMICA"]
ESTRATO = ["COD_CAMPUS", "UNIDAD", "CLASE", "COMPONENTE"]
resto = lambda d, b, k: (d * NB + b) * len(CLASES) + k


def clasificar(periodos_):
    usadas = pl[pl["PERIODO"].isin(periodos_) & pl["CON_SALA"]]
    fuera = usadas[~usadas.set_index(["EDIFICIO", "SALA"]).index.isin(planta.index)]
    comp = fuera.groupby(["EDIFICIO", "SALA"])["COD_COMPONENTE"].agg(lambda s: s.mode().iat[0]).reset_index()
    comp["CLASE"] = comp["EDIFICIO"].map(moda_edificio).fillna(comp["COD_COMPONENTE"].map(
        {"TEO": "SALA", "LAB": "LABORATORIO", "SIM": "LABORATORIO", "SSA": "LABORATORIO"}))
    clase = pd.concat([planta, comp.set_index(["EDIFICIO", "SALA"])["CLASE"]])
    return clase.where(clase.isin(CLASES), "FUERA")


def filas_clasificadas(periodos_filas, clase):
    x = pl[pl["PERIODO"].isin(periodos_filas) & pl["CON_SALA"]].copy()
    x["CLASE"] = clase.reindex(pd.MultiIndex.from_frame(x[["EDIFICIO", "SALA"]])).to_numpy()
    return x[x["CLASE"].isin(CLASES)]


def celda_de(df, campus_col="COD_CAMPUS_EDIFICIO"):
    return (df[campus_col].map(CAMP).to_numpy() * N_SC
            + resto(df["DIA"].map(DIA_I).to_numpy(), df["BLOQUE"].to_numpy(), df["CLASE"].map(CLA_I).to_numpy()))


def mapa_real(T):
    x = filas_clasificadas([T], clasificar([p for p in periodos if T - 200 <= p <= T]))
    suma = x.groupby(["EDIFICIO", "SALA", "DIA", "BLOQUE"])["PESO_SEMANAS"].transform("sum")
    x["OCUPADA"] = x["PESO_SEMANAS"] / suma * suma.clip(upper=1)
    return np.bincount(celda_de(x), weights=x["OCUPADA"].to_numpy(), minlength=N_CELDAS)


def secciones_fisicas(P, clase):
    x = filas_clasificadas([P], clase)
    meta = ph[ph["PERIODO"] == P][["NRC", "COD_LC", "N_INSCRITOS", "COD_MATERIA", "COD_COMPONENTE", "COD_CAMPUS"]]
    x = x.merge(meta.drop(columns="COD_COMPONENTE"), on="NRC")
    x["G0"] = x["COD_LC"].fillna("N" + x["NRC"].astype(str))
    cod, uniq = pd.factorize(x["G0"])
    x["_G"] = cod
    x["_SID"] = x.groupby(["EDIFICIO", "SALA", "DIA", "BLOQUE"]).ngroup()
    slot = x.groupby("_SID")["PESO_SEMANAS"].transform("sum") > 1 + 1e-6
    n_slots = x.drop_duplicates(["_G", "_SID"]).groupby("_G").size()
    e = x.loc[slot, ["_SID", "_G"]].drop_duplicates()
    pares = e.merge(e, on="_SID")
    pares = pares[pares["_G_x"] < pares["_G_y"]].groupby(["_G_x", "_G_y"]).size().rename("COMUN").reset_index()
    mayor = np.maximum(pares["_G_x"].map(n_slots), pares["_G_y"].map(n_slots))
    pares = pares[pares["COMUN"] >= 0.8 * mayor]
    adj = coo_matrix((np.ones(len(pares)), (pares["_G_x"].to_numpy(), pares["_G_y"].to_numpy())), shape=(len(uniq), len(uniq)))
    _, comp = connected_components(adj, directed=False)
    x["SEC"] = comp[cod]
    nrc = x.drop_duplicates(["SEC", "NRC"])
    dom = nrc.sort_values("N_INSCRITOS", ascending=False).drop_duplicates("SEC").set_index("SEC")
    clase_dom = (x.groupby(["SEC", "CLASE"])["PESO_SEMANAS"].sum().reset_index().sort_values("PESO_SEMANAS")
                  .drop_duplicates("SEC", keep="last").set_index("SEC")["CLASE"])
    sec = pd.DataFrame({"INSCRITOS": nrc.groupby("SEC")["N_INSCRITOS"].sum(), "COD_CAMPUS": dom["COD_CAMPUS"],
                        "MATERIA": dom["COD_MATERIA"], "UNIDAD": dom["COD_MATERIA"].map(UNIDAD_MAT).fillna("SIN UNIDAD"),
                        "COMPONENTE": dom["COD_COMPONENTE"], "CLASE": clase_dom})
    sec["VOL"] = sec["INSCRITOS"].clip(lower=1)
    filas = (x.groupby(["SEC", "EDIFICIO", "SALA", "DIA", "BLOQUE", "CLASE"])
              .agg(PESO=("PESO_SEMANAS", "sum"), CAMP=("COD_CAMPUS_EDIFICIO", "first")).reset_index())
    filas["PESO"] = filas["PESO"].clip(upper=1)
    return sec.reset_index(), filas


def elasticidad(T_cal):
    # β con los pares (P, P−100) de los 2 años anteriores a T_cal, del mismo nivel
    nivel = SUFIJOS[T_cal % 100]
    pares_p = [p for p in sorted(periodos) if T_cal - 200 <= p < T_cal and SUFIJOS.get(p % 100) == nivel and (p - 100) in periodos]
    clase = clasificar([p for p in periodos if p < T_cal])
    xs, ys, ws = [], [], []
    for P in pares_p:
        est = {}
        for Q in (P, P - 100):
            s, f = secciones_fisicas(Q, clase)
            s = s.assign(OCC=s["SEC"].map(f.groupby("SEC")["PESO"].sum()).fillna(0))
            est[Q] = s.groupby(ESTRATO)[["VOL", "OCC"]].sum()
        j = est[P].join(est[P - 100], lsuffix="_P", rsuffix="_A", how="inner")
        j = j[(j > 0).all(axis=1)]
        xs.append(np.log(j["VOL_P"] / j["VOL_A"])); ys.append(np.log(j["OCC_P"] / j["OCC_A"])); ws.append(j["OCC_A"])
    x, y, w = np.concatenate(xs), np.concatenate(ys), np.concatenate(ws)
    return float(np.clip((w * x * y).sum() / (w * x * x).sum(), 0, 1.5))


def estandar_real(s):
    s = s[s["INSCRITOS"] > 0]
    out = {}
    for key, g in s.groupby(["COD_CAMPUS", "COMPONENTE"]):
        ins, n = g["INSCRITOS"].sum(), len(g)
        q = {k: np.percentile(g["INSCRITOS"], k) for k in KS_REAL}
        k = min(KS_REAL, key=lambda k: abs(ins / max(q[k], 1) - n))
        out[key] = (k, q[k])
    return pd.DataFrame(out, index=["PERCENTIL", "ESTANDAR"]).T.rename_axis(["COD_CAMPUS", "COMPONENTE"])


RANGOS = ["0-20", "21-40", "41-60", "60 o más"]
TOPES = [20, 40, 60]
NR = len(RANGOS)


def rango_de(inscritos):
    # 0: ≤ 20 · 1: 21-40 · 2: 41-60 · 3: más de 60 (los mismos cortes de RANGO_CAPACIDAD de la planta)
    return np.digitize(np.asarray(inscritos, dtype=float), TOPES, right=True)


def tolerancia(secs, filas):
    cap = oferta.set_index(["EDIFICIO", "SALA"])["CAPACIDAD"]
    x = filas.merge(secs[["SEC", "INSCRITOS"]], on="SEC")
    sb = x.groupby(["EDIFICIO", "SALA", "DIA", "BLOQUE", "CLASE"])["INSCRITOS"].sum().reset_index()
    sb["CAP"] = cap.reindex(pd.MultiIndex.from_frame(sb[["EDIFICIO", "SALA"]])).to_numpy()
    sb = sb[sb["CAP"] > 0]
    return (sb["INSCRITOS"] / sb["CAP"]).groupby(sb["CLASE"]).quantile(0.9).clip(lower=1.0).to_dict()


def cap_seccion(filas):
    # capacidad de la sala más grande que usó cada sección en su período
    c = filas[["SEC", "EDIFICIO", "SALA"]].drop_duplicates()
    c["CAP"] = oferta.set_index(["EDIFICIO", "SALA"])["CAPACIDAD"].reindex(pd.MultiIndex.from_frame(c[["EDIFICIO", "SALA"]])).to_numpy()
    return c.groupby("SEC")["CAP"].max()


def tam_necesario(ins_proy, ins_base, cap_sec, tol):
    # capacidad que necesita la sección: sus inscritos proyectados ÷ el sobrecupo que ya tenía en su sala (o la tolerancia, si es mayor).
    # Así el período base nunca "necesita" un rango mayor que el de su propia sala: solo el crecimiento genera falta.
    sobre = np.where(np.asarray(cap_sec, dtype=float) > 0, np.asarray(ins_base, dtype=float) / np.asarray(cap_sec, dtype=float), 0.0)
    return np.asarray(ins_proy, dtype=float) / np.maximum(np.asarray(tol, dtype=float), np.nan_to_num(sobre))


def remuestrear(secs, objetivo, columna, rng, k_max=10):
    # secs: secciones base con EST y la medida a acumular (VOL o 1 por sección); objetivo: EST → meta.
    # Primero la base completa en orden aleatorio, después copias; se toma hasta acercarse a la meta.
    b = secs.merge(objetivo.rename("OBJ"), left_on="EST", right_index=True)
    tot = b.groupby("EST")[columna].transform("sum")
    b["K"] = np.minimum(k_max, np.ceil(b["OBJ"] / tot) + 1).astype(int)
    rep = b.loc[b.index.repeat(b["K"])].copy()
    rep["COPIA"] = rep.groupby(level=0).cumcount()
    rep = rep.reset_index(drop=True)
    rep["KEY"] = rep["COPIA"] + rng.random(len(rep))
    o = rep.sort_values(["EST", "KEY"])
    acum = o.groupby("EST")[columna].cumsum()
    return o[((acum - o[columna] / 2) < o["OBJ"]).to_numpy()]


def asignar_salas(elegidas, filas, inventario, tol):
    # D18 · Una sección original (copia 0) queda en su sala solo si sus inscritos PROYECTADOS caben: hasta
    # capacidad × tolerancia de su clase, o hasta lo que ya tenía en el período base (el sobrecupo histórico se respeta).
    # Si no cabe, y las copias siempre, buscan una sala libre de la misma clase con capacidad × tolerancia ≥ inscritos
    # proyectados, la más chica que sirva, primero en el mismo edificio y después en el mismo campus. Lo que no cabe es déficit.
    carga, salida = {}, []
    cap = inventario.set_index(["EDIFICIO", "SALA"])["CAPACIDAD"].to_dict()
    por_clase = {k: g.sort_values("CAPACIDAD") for k, g in inventario.groupby(["COD_CAMPUS_EDIFICIO", "CLASE_MAPA"])}
    filas_sec = {k: g for k, g in filas.groupby("SEC")}
    for i, r in enumerate(elegidas.sort_values("COPIA").itertuples(index=False)):
        fs = filas_sec.get(r.SEC)
        if fs is None:
            continue
        for (edif, sala, camp, clase), g in fs.groupby(["EDIFICIO", "SALA", "CAMP", "CLASE"]):
            slots = list(zip(g["DIA"], g["BLOQUE"], g["PESO"]))
            libre = lambda e, s_: all(carga.get((e, s_, d, b), 0) + p <= 1 + 1e-9 for d, b, p in slots)
            t = tol.get(clase, 1.0)
            c0 = cap.get((edif, sala))
            cabe0 = c0 is None or r.INSCRITOS_EST <= max(c0 * t, r.INSCRITOS)
            destino = (edif, sala) if (r.COPIA == 0 and cabe0 and libre(edif, sala)) else None
            if destino is None:
                cand = por_clase.get((camp, clase))
                if cand is not None:
                    cand = cand[cand["CAPACIDAD"].fillna(0) * t >= r.INSCRITOS_EST]
                    cand = pd.concat([cand[cand["EDIFICIO"] == edif], cand[cand["EDIFICIO"] != edif]])
                    destino = next(((e, s_) for e, s_ in zip(cand["EDIFICIO"], cand["SALA"]) if libre(e, s_)), None)
            for d, b, p in slots:
                if destino is not None:
                    carga[(destino[0], destino[1], d, b)] = carga.get((destino[0], destino[1], d, b), 0) + p
                salida.append((i, r.COD_CAMPUS, r.UNIDAD, r.MATERIA, r.COMPONENTE, r.INSCRITOS_EST, camp, clase,
                               destino[0] if destino else None, destino[1] if destino else None,
                               cap.get(destino) if destino else np.nan, RANGOS[int(rango_de(tam_necesario(r.INSCRITOS_EST, r.INSCRITOS, c0 if c0 else np.nan, t)))],
                               d, b, p, destino is None))
    out = pd.DataFrame(salida, columns=["SECCION_SINTETICA", "COD_CAMPUS", "UNIDAD", "MATERIA", "COMPONENTE", "INSCRITOS_EST",
                                        "COD_CAMPUS_EDIFICIO", "CLASE", "EDIFICIO", "SALA", "CAPACIDAD_SALA", "RANGO_DEMANDADO",
                                        "DIA", "BLOQUE", "PESO_SEMANAS", "DEFICIT"])
    out["SOBRECUPO"] = ~out["DEFICIT"] & (out["INSCRITOS_EST"] > out["CAPACIDAD_SALA"])       # cabe por tolerancia, no por capacidad nominal
    return out

## 3 · Volumen proyectado

El volumen entra como **razón de crecimiento por tipo** (campus × materia × componente). Esa razón se aplica a los inscritos del último período real del mismo sufijo, que es la base.

**2027 · escenario base = fuente adoptada en el banco por nivel (D15): UG = TEN, AV = EUR**
- **TEN:** crecimiento de los inscritos del campus entre el período base y el del año anterior (mismo sufijo). En AV no se usa: en 2026 la matrícula AV se trasladó de C02 a C05 (×0,4 y ×1,7), y la tendencia lo extrapolaría como si se repitiera.
- **EUR:** Σ P50 ÷ Σ `ins_lag_1y`, dentro de Euribia.
- **PER:** razón 1.
- **Bajo / alto = base × (Σ P10 ÷ Σ P50) / (Σ P90 ÷ Σ P50)** de Euribia por tipo: Euribia queda como **banda**. Los tipos que Euribia no proyecta no tienen banda, y OCS queda fuera.
- Si Euribia fuera la fuente, **no se usaría `P50_ESPERADO`** (P50 × `PROB_SE_DICTA`). Descuenta los cursos que podrían cerrarse sin sumar los cursos nuevos que Euribia no ve, y en el agregado hunde el volumen de 304 mil a 250 mil inscritos en dos pasos. El P50 condicional, en cambio, da cerca de −1 %, coherente con la historia.
- Bajo y alto **suman cuantiles por tipo**: equivalen a que todos los cursos caigan o suban a la vez, y son más extremos que un cuantil del total.

**2028–2031 · Ariadna (pseudo-backtest §7 del banco)**
- Razón = razón base 2027 × matrícula **anual** de Ariadna del año T (`n_bajo` / `n` / `n_alto`) ÷ matrícula anual 2027 (`n`), por carrera × sede (`MATERIA` = `cod_carrera`). Los departamentos de servicio usan la de su sede.
- Es anual porque la matrícula se registra casi entera en el período 10: por semestre, el 20 y el 15 serían ruido.
- **Se encadena a 2027** para que el largo plazo parta del escenario base 2027 sin saltos.
- **Bajo / alto** = la banda de Ariadna (`n_bajo` / `n_alto`), ensanchada alrededor del base por el factor de `ARIADNA_ERROR_HORIZONTE.parquet` cuando su backtest muestra que la banda no cubre el error real en el 80 % de los casos. Ariadna midió su error hasta H+3; los años siguientes usan el factor de H+3.

Las razones se acotan entre 0,25 y 4.

In [38]:
TIPO = ["COD_CAMPUS", "MATERIA", "COMPONENTE"]
ins_base = {s: ph[ph["PERIODO"] == B].rename(columns={"COD_MATERIA": "MATERIA", "COD_COMPONENTE": "COMPONENTE"})
                 .groupby(TIPO)["N_INSCRITOS"].sum() for s, B in BASE.items()}
ap = ari[ari["tipo"] == "PROYECTADO"].assign(ANIO=lambda d: d["periodo"] // 100)
ari_anual = ap.groupby(["ANIO", "cod_carrera", "sede"])[["n_bajo", "n", "n_alto"]].sum()
ari_sede = ap.groupby(["ANIO", "sede"])[["n_bajo", "n", "n_alto"]].sum()
COL_ARI = {"bajo": "n_bajo", "base": "n", "alto": "n_alto"}
COL_EUR = {"bajo": "P10", "base": "P50", "alto": "P90"}


def crecimiento_ariadna(anio, col, base_t):
    # matrícula anual del año / matrícula anual de ANIO_CORTO, por carrera × sede; si la materia no es carrera, por sede
    idx_m = pd.MultiIndex.from_arrays([base_t.index.get_level_values("MATERIA"), base_t.index.get_level_values("COD_CAMPUS")])
    sede = base_t.index.get_level_values("COD_CAMPUS")
    r_m = ari_anual.loc[anio, col].reindex(idx_m).to_numpy() / ari_anual.loc[ANIO_CORTO, "n"].reindex(idx_m).to_numpy()
    r_s = ari_sede.loc[anio, col].reindex(sede).to_numpy() / ari_sede.loc[ANIO_CORTO, "n"].reindex(sede).to_numpy()
    ok_m = np.isfinite(r_m) & (r_m > 0)
    return pd.Series(np.where(ok_m, r_m, np.where(np.isfinite(r_s) & (r_s > 0), r_s, 1.0)), index=base_t.index), ok_m.mean()


filas_vol, razon_corto, cobertura = [], {}, {}
for T, plazo in OBJETIVOS:                       # los objetivos de corto plazo van primero
    s = T % 100; B = BASE[s]; base_t = ins_base[s]
    for esc in ESCENARIOS:
        if plazo == "corto":
            e = eur[(eur["PERIODO"] == T) & ~eur["COMPONENTE"].isin(COMPONENTES_SIN_HORARIO) & (eur["ins_lag_1y"] > 0)]
            g = e.groupby(["CAMPUS", "MATERIA", "COMPONENTE"])[["P10", "P50", "P90", "ins_lag_1y"]].sum()
            g.index.names = TIPO
            fuente = FUENTE_VOLUMEN[SUFIJOS[s]]
            if fuente == "EUR":
                r_base = (g["P50"] / g["ins_lag_1y"]).reindex(base_t.index).fillna(1.0)
            elif fuente == "TEN":
                camp = ph[ph["PERIODO"].isin([B, B - 100])].groupby(["PERIODO", "COD_CAMPUS"])["N_INSCRITOS"].sum().unstack(0)
                r_base = pd.Series(base_t.index.get_level_values("COD_CAMPUS").map(camp[B] / camp[B - 100]), index=base_t.index).fillna(1.0)
            else:
                r_base = pd.Series(1.0, index=base_t.index)
            banda = (g[COL_EUR[esc]] / g["P50"].where(g["P50"] > 0)).reindex(base_t.index).fillna(1.0)
            razon = (r_base * banda).clip(RAZON_MIN, RAZON_MAX)
            cobertura[T] = float(base_t[(g["ins_lag_1y"].reindex(base_t.index).fillna(0) > 0).to_numpy()].sum() / base_t.sum())
            if esc == "base":
                razon_corto[s] = razon
        else:
            crec, cobertura[T] = crecimiento_ariadna(T // 100, COL_ARI[esc], base_t)
            if esc != "base":
                crec_b, _ = crecimiento_ariadna(T // 100, "n", base_t)
                f = ENSANCHE_ARI.get((SUFIJOS[s], min(T // 100 - 2026, H_MAX_ARI)), 1.0)     # h de Ariadna: años desde 2026
                crec = crec_b + f * (crec - crec_b)
            razon = (razon_corto[s] * crec).clip(RAZON_MIN, RAZON_MAX)
        filas_vol.append(pd.DataFrame({"PERIODO": T, "PLAZO": plazo, "ESCENARIO": esc, "BASE": B,
                                       "INSCRITOS_BASE": base_t, "RAZON": razon}).reset_index())
vol = pd.concat(filas_vol, ignore_index=True)
vol["INSCRITOS_PROY"] = vol["INSCRITOS_BASE"] * vol["RAZON"]
vol["H"] = vol["PERIODO"].map(horizonte)
vol["VALIDADO"] = vol["PERIODO"].map(validado)
vol.to_parquet(BBDD / "VOLUMEN_PROYECTADO.parquet", index=False)
print("cobertura (corto: % de inscritos base con banda de Euribia · largo: % de tipos con carrera en Ariadna):")
print({T: round(c, 3) for T, c in cobertura.items()})
t = vol.groupby(["PERIODO", "ESCENARIO"])["INSCRITOS_PROY"].sum().unstack("ESCENARIO")[ESCENARIOS]
t.insert(0, "BASE_REAL", vol.drop_duplicates(["PERIODO"] + TIPO).groupby("PERIODO")["INSCRITOS_BASE"].sum())
t.round(0)

C:\Users\usuario\AppData\Local\Temp\ipykernel_52588\380145281.py:15: RuntimeWarning: invalid value encountered in divide
  r_m = ari_anual.loc[anio, col].reindex(idx_m).to_numpy() / ari_anual.loc[ANIO_CORTO, "n"].reindex(idx_m).to_numpy()
C:\Users\usuario\AppData\Local\Temp\ipykernel_52588\380145281.py:15: RuntimeWarning: invalid value encountered in divide
  r_m = ari_anual.loc[anio, col].reindex(idx_m).to_numpy() / ari_anual.loc[ANIO_CORTO, "n"].reindex(idx_m).to_numpy()
C:\Users\usuario\AppData\Local\Temp\ipykernel_52588\380145281.py:15: RuntimeWarning: invalid value encountered in divide
  r_m = ari_anual.loc[anio, col].reindex(idx_m).to_numpy() / ari_anual.loc[ANIO_CORTO, "n"].reindex(idx_m).to_numpy()
C:\Users\usuario\AppData\Local\Temp\ipykernel_52588\380145281.py:15: RuntimeWarning: invalid value encountered in divide
  r_m = ari_anual.loc[anio, col].reindex(idx_m).to_numpy() / ari_anual.loc[ANIO_CORTO, "n"].reindex(idx_m).to_numpy()
C:\Users\usuario\AppData\Local\Temp\ipykerne

cobertura (corto: % de inscritos base con banda de Euribia · largo: % de tipos con carrera en Ariadna):
{202710: 1.0, 202720: 0.975, 202705: 0.999, 202715: 0.998, 202725: 0.866, 202810: 0.657, 202820: 0.699, 202805: 0.585, 202815: 0.632, 202825: 0.635, 202910: 0.657, 202920: 0.699, 202905: 0.585, 202915: 0.632, 202925: 0.635, 203010: 0.657, 203020: 0.699, 203005: 0.585, 203015: 0.632, 203025: 0.635, 203110: 0.657, 203120: 0.699, 203105: 0.585, 203115: 0.632, 203125: 0.635}


ESCENARIO,BASE_REAL,bajo,base,alto
PERIODO,,,,
202705,46481,34187.0,46124.0,62640.0
202710,329627,249435.0,345663.0,466461.0
202715,44367,32388.0,43746.0,61149.0
202720,309211,230076.0,327739.0,453539.0
202725,40489,32072.0,40119.0,53802.0
202805,46481,38024.0,48845.0,59684.0
202810,329627,331458.0,364946.0,398469.0
202815,44367,35963.0,46337.0,56702.0
202820,309211,314646.0,346047.0,377476.0


## 4 · Número de secciones · estándar real (H2)

- Por sede × componente, el **estándar real** es el percentil P5…P95 del tamaño de las secciones físicas del período base cuyo número implícito de secciones reproduce las que se programaron.
- Las secciones del objetivo son los inscritos físicos proyectados (los de la base × la razón de §3) divididos por ese estándar.

In [39]:
_cache = {}
def base_fisica(s):
    if s not in _cache:
        B = BASE[s]
        _cache[s] = secciones_fisicas(B, clasificar([p for p in periodos if B - 200 <= p <= B]))
    return _cache[s]

ESTANDAR = {s: estandar_real(base_fisica(s)[0]) for s in SUFIJOS}
filas_n = []
for (T, esc), g in vol.groupby(["PERIODO", "ESCENARIO"]):
    s = T % 100
    secs, _ = base_fisica(s)
    r = secs.merge(g[TIPO + ["RAZON"]], on=TIPO, how="left").fillna({"RAZON": 1.0})
    r["INS_PROY"] = r["INSCRITOS"] * r["RAZON"]
    sc = r.groupby(["COD_CAMPUS", "COMPONENTE"]).agg(INSCRITOS_PROY=("INS_PROY", "sum"), SECCIONES_BASE=("SEC", "size"))
    sc = sc.join(ESTANDAR[s])
    sc["SECCIONES_PROY"] = np.maximum(1, np.round(sc["INSCRITOS_PROY"] / sc["ESTANDAR"].clip(lower=1))).fillna(sc["SECCIONES_BASE"]).astype(int)
    filas_n.append(sc.reset_index().assign(PERIODO=T, ESCENARIO=esc, PLAZO=g["PLAZO"].iat[0]))
secciones_proy = pd.concat(filas_n, ignore_index=True)
secciones_proy.to_parquet(BBDD / "SECCIONES_PROYECTADAS.parquet", index=False)
secciones_proy.groupby(["PERIODO", "ESCENARIO"])[["SECCIONES_BASE", "SECCIONES_PROY"]].sum().unstack("ESCENARIO").head(10)

SECCIONES_BASE             SECCIONES_PROY             
ESCENARIO           alto  bajo  base           alto  bajo   base
PERIODO                                                         
202705              1353  1353  1353           1733   943   1299
202710              9397  9397  9397          13327  7115   9885
202715              1280  1280  1280           1639   891   1225
202720              8954  8954  8954          12955  6549   9389
202725              1170  1170  1170           1366   865   1103
202805              1353  1353  1353           1672  1079   1375
202810              9397  9397  9397          11399  9478  10434
202815              1280  1280  1280           1583  1012   1297
202820              8954  8954  8954          10817  9014   9915
202825              1170  1170  1170           1425   911   1171

## 5 · Mapa con C1

Para cada período objetivo y escenario:
- **naive** = mapa real del período base (contado por sala);
- **G1b** = secciones físicas del período base remuestreadas por estrato (campus × unidad × clase × componente) hasta volumen_base × razón^β;
- **mapa = w · naive + (1 − w) · G1b**, con el w de la variante adoptada en el banco, por nivel (w = 1 es el naive).

β se calibra una vez por sufijo con los pares reales previos al primer objetivo.

**Banda calibrada (H5, banco §5b):**
- punto = media de las simulaciones de C1;
- P10 / P50 / P90 = punto + q10 / q50 / q90 × √(punto + 1), con los cuantiles conformales de `BANDA_CUANTILES.parquet` (modelo adoptado × nivel × clase × tramo del punto, grano celda);
- las simulaciones de §7 se rehacen sorteando residuos de esos mismos cuantiles, para que la probabilidad de saturar sea coherente con la banda;
- en las celdas donde el campus no tiene espacios activos de esa clase, la ocupación es 0 (restricción física). Sin ella, el cuantil 90 positivo del tramo 0–5 pondría ~2 salas en celdas imposibles.

**Rango de capacidad (D18):**
- los inscritos proyectados de cada sección son los de la base × razón del estrato^(1 − β): G1b agrega secciones al ritmo razón^β, así que el resto del crecimiento engorda las secciones;
- la capacidad que necesita = inscritos proyectados ÷ el mayor entre la tolerancia de su clase y el sobrecupo que ya tenía en su sala de la base;
- la **oferta efectiva** por rango (desde 21-40) es el mayor entre el inventario activo y lo que el período base ya usa: hay salas en uso que no están activas en la planta o que la planta clasifica distinto. Con ambas reglas, el período base nunca queda en alerta y solo el crecimiento genera falta. Sin esto, los laboratorios que ya funcionan sobre su capacidad nominal (rotaciones, grupos) aparecían pidiendo laboratorios de 60 o más, que casi no existen;
- con ese tamaño, cada sección cuenta en el rango que necesita, no en el de la sala donde estaba;
- el naive se reparte por rango con los tamaños del período base;
- la mezcla por rango de cada celda sale de C1 (w · naive + (1 − w) · G1b) y reparte las simulaciones calibradas de la celda; la banda sigue calibrada en el grano celda.

**Largo plazo (D17, banco §7):**
- **Modelo del año (D19, 28-10):** 2027 con C1; 2028–2031 con **C1E**, el naive escalado por campus × clase con el crecimiento de salas que implica el objetivo de G1b, y el w de C1E en el banco. En el pseudo-backtest C1E corrige el sesgo de C1 (H+4: −7,6 % → −2,7 %) y mejora el MASE desde H+3 (0,959 / 0,962 contra 0,995 / 0,993), pero en H+2 lo empeora en 0,011 (1,012 contra 1,001). Antes (27-10) era C1 con el w del banco en todos los años. En el pseudo-backtest C1 gana o pierde contra el naive de la base por ±0,02, así que elegirlo por horizonte alternaba C1 y naive y dejaba 2029 sin crecer. La ventaja de C1 es marginal y queda como aviso.
- **Banda:** los cuantiles de 1 año se ensanchan alrededor de la mediana por k_h: q' = q50 + k_h · (q − q50). k_h está medido hasta H+4; el de H+5 (2031) es extrapolado.

In [40]:
BETA = {s: elasticidad(BASE[s] + 100) for s in SUFIJOS}
print("β por sufijo:", {s: round(b, 3) for s, b in BETA.items()})
NAIVE = {s: mapa_real(BASE[s]) for s in SUFIJOS}
rng = np.random.default_rng(SEMILLA)
# D18 · tolerancia por clase, oferta por rango y reparto del naive por rango
TOL = {s: tolerancia(*base_fisica(s)) for s in SUFIJOS}
print("tolerancia de capacidad (P90 de inscritos ÷ capacidad en la base):", {s: {k: round(v, 2) for k, v in t.items()} for s, t in TOL.items()})
_inv = oferta[oferta["ACTIVO"] & oferta["CLASE_MAPA"].isin(CLASES)]
_cap_r = _inv.assign(R=rango_de(_inv["CAPACIDAD"].fillna(0))).groupby(["COD_CAMPUS_EDIFICIO", "CLASE_MAPA", "R"]).size()
CAP_RANGO = np.array([[_cap_r.get((c, k, r), 0) for r in range(NR)] for c in CAMP for _ in DIAS for _ in range(NB) for k in CLASES], dtype=float)
CAP_ACUM = CAP_RANGO[:, ::-1].cumsum(1)[:, ::-1]                 # espacios con rango ≥ r


def parte_por_rango(m):
    tot = m.sum(1, keepdims=True)
    return np.where(tot > 0, m / np.where(tot > 0, tot, 1), np.eye(NR)[0])


def naive_por_rango(s):
    secs, filas = base_fisica(s)
    f = filas.merge(secs[["SEC", "INSCRITOS"]], on="SEC")
    rr = rango_de(tam_necesario(f["INSCRITOS"], f["INSCRITOS"], f["SEC"].map(cap_seccion(filas)), f["CLASE"].map(TOL[s]).fillna(1.0)))
    cel = celda_de(f.rename(columns={"CAMP": "COD_CAMPUS_EDIFICIO"}))
    m = np.bincount(cel * NR + rr, weights=f["PESO"].to_numpy(), minlength=N_CELDAS * NR).reshape(N_CELDAS, NR)
    return NAIVE[s][:, None] * parte_por_rango(m)


NAIVE_RANGO = {s: naive_por_rango(s) for s in SUFIJOS}
# Oferta efectiva por rango (r ≥ 21-40) = max(inventario activo, lo que el período base ya usa). Hay salas usadas que no están
# activas en la planta o que la planta clasifica distinto; si la base cabe, esa oferta existe. Así el período base real
# nunca queda en alerta y solo el crecimiento la genera. El rango 0-20 (la celda completa) no se toca: es la PROB_SATURA de siempre.
CAP_ACUM_S = {}
for s_ in SUFIJOS:
    _acum = NAIVE_RANGO[s_][:, ::-1].cumsum(1)[:, ::-1]
    CAP_ACUM_S[s_] = np.column_stack([CAP_ACUM[:, 0], np.maximum(CAP_ACUM[:, 1:], _acum[:, 1:])])
    print(f"base {BASE[s_]}: celdas-rango con oferta efectiva > inventario activo =", int((_acum[:, 1:] > CAP_ACUM[:, 1:] + 1e-6).sum()))
PARTE_RANGO = {}
bq = pd.read_parquet(BBDD / "BANDA_CUANTILES.parquet")
COLS_Q = [c for c in bq.columns if c.startswith("Q")]
NIV_Q = np.array([int(c[1:]) / 100 for c in COLS_Q])
TRAMOS = [0, 5, 20, 50, np.inf]
CLASE_CELDA = np.array(CLASES)[np.arange(N_CELDAS) % len(CLASES)]
# Celdas físicamente posibles: el campus tiene espacios activos de esa clase. En las demás la ocupación es 0.
_act = oferta[oferta["ACTIVO"] & oferta["CLASE_MAPA"].isin(CLASES)].groupby(["COD_CAMPUS_EDIFICIO", "CLASE_MAPA"]).size()
POSIBLE = np.array([_act.get((c, k), 0) > 0 for c in CAMP for _ in DIAS for _ in range(NB) for k in CLASES])


def cuantiles_celda(nivel, punto, h=1):
    m = (MODELO_LP if h >= ESCALAR_DESDE_H else MODELO_ADOPTADO)[nivel] or "C1-PER"
    t = bq[(bq["MODELO"] == m) & (bq["NIVEL"] == nivel) & (bq["GRANO"] == "celda")].set_index(["CLASE", "TRAMO"])[COLS_Q]
    tramo = pd.cut(punto, TRAMOS, right=False, labels=False)
    return t.reindex(pd.MultiIndex.from_arrays([CLASE_CELDA, tramo])).to_numpy()    # celdas × cuantiles
mapas, sims_guardadas = [], {}
for (T, esc), g in vol.groupby(["PERIODO", "ESCENARIO"]):
    s = T % 100; nivel = SUFIJOS[s]; plazo = g["PLAZO"].iat[0]
    secs, filas = base_fisica(s)
    r = secs.merge(g[TIPO + ["RAZON"]], on=TIPO, how="left").fillna({"RAZON": 1.0})
    r["EST"] = r[ESTRATO].astype(str).agg("|".join, axis=1)
    vb = r.groupby("EST")["VOL"].sum()
    razon_est = (r["VOL"] * r["RAZON"]).groupby(r["EST"]).sum() / vb
    objetivo = vb * razon_est ** BETA[s]
    # D19 · factor de escala del naive por campus × clase: crecimiento de salas que implica el objetivo (1 en 2027)
    h = horizonte(T)
    g_cel = np.ones(N_CELDAS)
    if h >= ESCALAR_DESDE_H:
        _e = r.drop_duplicates("EST").set_index("EST")[["COD_CAMPUS", "CLASE"]].reindex(vb.index)
        _crece = objetivo.groupby([_e["COD_CAMPUS"], _e["CLASE"]]).sum() / vb.groupby([_e["COD_CAMPUS"], _e["CLASE"]]).sum()
        _d, _b = np.repeat(np.arange(len(DIAS)), NB), np.tile(np.arange(NB), len(DIAS))
        for (c_, k_), v_ in _crece.items():
            if c_ in CAMP and k_ in CLA_I and np.isfinite(v_):
                g_cel[CAMP[c_] * N_SC + resto(_d, _b, CLA_I[k_])] = v_
    f = filas.sort_values("SEC").reset_index(drop=True)
    f_sec, f_peso = f["SEC"].to_numpy(), f["PESO"].to_numpy()
    f_cel = celda_de(f.rename(columns={"CAMP": "COD_CAMPUS_EDIFICIO"}))
    # D18 · rango que necesita cada sección: inscritos × razón^(1 − β) ÷ tolerancia de su clase
    tam = tam_necesario(r["INSCRITOS"].to_numpy() * r["EST"].map(razon_est).to_numpy() ** (1 - BETA[s]), r["INSCRITOS"],
                        r["SEC"].map(cap_seccion(filas)), r["CLASE"].map(TOL[s]).fillna(1.0))
    f_rango = f["SEC"].map(pd.Series(rango_de(tam), index=r["SEC"].to_numpy())).fillna(0).astype(int).to_numpy()
    sims = np.zeros((N_SIM[plazo], N_CELDAS))
    sims_r = np.zeros((N_SIM[plazo], N_CELDAS * NR))
    for k in range(N_SIM[plazo]):
        el = remuestrear(r, objetivo, "VOL", rng)
        ini = np.searchsorted(f_sec, el["SEC"].to_numpy(), "left"); fin = np.searchsorted(f_sec, el["SEC"].to_numpy(), "right")
        largo = fin - ini
        idx = np.repeat(ini, largo) + (np.arange(largo.sum()) - np.repeat(np.cumsum(largo) - largo, largo))
        sims[k] = np.bincount(f_cel[idx], weights=f_peso[idx], minlength=N_CELDAS)
        sims_r[k] = np.bincount(f_cel[idx] * NR + f_rango[idx], weights=f_peso[idx], minlength=N_CELDAS * NR)
    w = (W_NAIVE_LP if h >= ESCALAR_DESDE_H else W_NAIVE)[nivel]      # C1 en 2027, C1E desde 2028 (D19)
    comb = w * (NAIVE[s] * g_cel)[None, :] + (1 - w) * sims
    PARTE_RANGO[(T, esc)] = parte_por_rango(w * NAIVE_RANGO[s] * g_cel[:, None] + (1 - w) * sims_r.reshape(-1, N_CELDAS, NR).mean(0))
    punto = comb.mean(0)
    Q = cuantiles_celda(nivel, punto, h)
    if h > 1:
        q50 = Q[:, [list(NIV_Q.round(2)).index(0.50)]]
        Q = q50 + K_H.get((nivel, h), 1.0) * (Q - q50)
    esc_c = np.sqrt(punto + 1)
    p10, p50, p90 = (np.where(POSIBLE, np.clip(punto + Q[:, list(NIV_Q.round(2)).index(a)] * esc_c, 0, None), 0) for a in (0.10, 0.50, 0.90))
    # conjunto calibrado para §7: un residuo por simulación y celda, interpolado en la grilla q05…q95
    u = rng.uniform(NIV_Q[0], NIV_Q[-1], size=comb.shape)
    e = np.stack([np.interp(u[:, j], NIV_Q, Q[j]) for j in range(N_CELDAS)], axis=1)
    sims_guardadas[(T, esc)] = np.where(POSIBLE[None, :], np.clip(punto[None, :] + e * esc_c[None, :], 0, None), 0)
    cel = np.arange(N_CELDAS)
    camp_i, rest = np.divmod(cel, N_SC); dia_i, rest2 = np.divmod(rest, NB * len(CLASES)); blo, cla_i = np.divmod(rest2, len(CLASES))
    df = pd.DataFrame({"PERIODO": T, "ESCENARIO": esc, "PLAZO": plazo, "H": h, "VALIDADO": validado(T),
                       "COD_CAMPUS_EDIFICIO": np.array(list(CAMP))[camp_i], "DIA": np.array(DIAS)[dia_i], "BLOQUE": blo.astype("int64"),
                       "CLASE": np.array(CLASES)[cla_i], "OCUPACION_PUNTUAL": punto, "OCUPACION_P10": p10, "OCUPACION_P50": p50, "OCUPACION_P90": p90,
                       "W_NAIVE": w, "BETA": BETA[s], "MODELO": "C1E" if h >= ESCALAR_DESDE_H else "C1",
                       "ESCALA_NAIVE": g_cel})
    mapas.append(df[(df[["OCUPACION_P10", "OCUPACION_P50", "OCUPACION_P90"]] > 0).any(axis=1)])
mapa = pd.concat(mapas, ignore_index=True)
mapa.to_parquet(BBDD / "MAPA_PROYECTADO.parquet", index=False)
mapa.groupby(["PERIODO", "ESCENARIO"])["OCUPACION_P50"].sum().unstack("ESCENARIO").round(0).head(10)

β por sufijo: {10: 0.821, 20: 0.799, 5: 0.69, 15: 0.69, 25: 0.685}
tolerancia de capacidad (P90 de inscritos ÷ capacidad en la base): {10: {'LABORATORIO': 1.33, 'SALA': 1.0}, 20: {'LABORATORIO': 1.32, 'SALA': 1.0}, 5: {'LABORATORIO': 1.0, 'SALA': 1.0}, 15: {'LABORATORIO': 1.0, 'SALA': 1.0}, 25: {'LABORATORIO': 1.0, 'SALA': 1.0}}
base 202610: celdas-rango con oferta efectiva > inventario activo = 20
base 202620: celdas-rango con oferta efectiva > inventario activo = 74
base 202605: celdas-rango con oferta efectiva > inventario activo = 0
base 202615: celdas-rango con oferta efectiva > inventario activo = 0
base 202625: celdas-rango con oferta efectiva > inventario activo = 0


ESCENARIO,alto,bajo,base
PERIODO,,,
202705,4050.0,2604.0,3239.0
202710,27848.0,24898.0,26270.0
202715,3977.0,2508.0,3178.0
202720,26359.0,23398.0,24768.0
202725,3846.0,2708.0,3235.0
202805,3983.0,2912.0,3464.0
202810,28635.0,24571.0,26636.0
202815,3853.0,2814.0,3344.0
202820,26954.0,23292.0,25128.0


## 6 · Programación sintética explícita

Para el **escenario base** de cada período:
- las secciones físicas del período base se remuestrean por sede × componente **hasta el número de secciones de §4** (estándar real);
- cada copia lleva sus inscritos escalados por la razón de su tipo;
- se ubican con la asignación a sala libre equivalente.

Cada fila es una sección sintética en un casillero (día × bloque), con su sala asignada o marcada como déficit. Los escenarios bajo y alto solo cambian el número de secciones de §4.

**D18:** la asignación usa los inscritos **proyectados** de cada sección (antes usaba los del período base) y la tolerancia de capacidad de su clase. Cada fila trae `CAPACIDAD_SALA`, `RANGO_DEMANDADO` y `SOBRECUPO` (cabe por tolerancia, pero supera la capacidad nominal).

In [41]:
inventario = oferta[oferta["ACTIVO"] & oferta["CLASE_MAPA"].isin(CLASES)]
progs = []
for T, plazo in OBJETIVOS:
    s = T % 100
    secs, filas = base_fisica(s)
    g = vol[(vol["PERIODO"] == T) & (vol["ESCENARIO"] == "base")]
    r = secs.merge(g[TIPO + ["RAZON"]], on=TIPO, how="left").fillna({"RAZON": 1.0})
    r["INSCRITOS_EST"] = np.round(r["INSCRITOS"] * r["RAZON"]).astype(int)
    r["EST"] = r["COD_CAMPUS"] + "|" + r["COMPONENTE"]
    r["UNO"] = 1
    meta_n = secciones_proy[(secciones_proy["PERIODO"] == T) & (secciones_proy["ESCENARIO"] == "base")]
    objetivo = pd.Series(meta_n["SECCIONES_PROY"].to_numpy(), index=meta_n["COD_CAMPUS"] + "|" + meta_n["COMPONENTE"])
    el = remuestrear(r, objetivo, "UNO", np.random.default_rng(SEMILLA))
    prog = asignar_salas(el, filas, inventario, TOL[s])
    progs.append(prog.assign(PERIODO=T, PLAZO=plazo, H=horizonte(T), VALIDADO=validado(T), NIVEL=SUFIJOS[s]))
    print(T, f"secciones {el.shape[0]:,} · déficit {prog.loc[prog['DEFICIT'], 'PESO_SEMANAS'].sum() / prog['PESO_SEMANAS'].sum():.1%}".replace(",", "."))
programacion = pd.concat(progs, ignore_index=True)
programacion["BLOQUE"] = programacion["BLOQUE"].astype("int64")
programacion.to_parquet(BBDD / "PROGRAMACION_SINTETICA.parquet", index=False)

202710 secciones 9.885 · déficit 5.3%
202720 secciones 9.389 · déficit 4.6%
202705 secciones 1.299 · déficit 0.2%
202715 secciones 1.225 · déficit 0.0%
202725 secciones 1.103 · déficit 0.0%
202810 secciones 10.434 · déficit 7.6%
202820 secciones 9.915 · déficit 6.8%
202805 secciones 1.375 · déficit 0.2%
202815 secciones 1.297 · déficit 0.0%
202825 secciones 1.171 · déficit 0.0%
202910 secciones 11.015 · déficit 10.7%
202920 secciones 10.463 · déficit 9.5%
202905 secciones 1.448 · déficit 0.4%
202915 secciones 1.369 · déficit 0.0%
202925 secciones 1.232 · déficit 0.0%
203010 secciones 11.626 · déficit 13.9%
203020 secciones 11.040 · déficit 12.5%
203005 secciones 1.523 · déficit 0.4%
203015 secciones 1.436 · déficit 0.0%
203025 secciones 1.293 · déficit 0.1%
203110 secciones 12.263 · déficit 17.4%
203120 secciones 11.652 · déficit 15.7%
203105 secciones 1.594 · déficit 0.5%
203115 secciones 1.501 · déficit 0.1%
203125 secciones 1.352 · déficit 0.0%


## 7 · Ocupación contra la oferta

Por celda (campus del edificio × día × bloque × clase):
- **tasa de uso** = ocupación P50 ÷ espacios activos de esa clase en el campus;
- **probabilidad de saturar** = % de simulaciones calibradas (§5) en que la ocupación supera los espacios activos;
- **déficit** = casilleros de la programación explícita sin sala equivalente libre.

**Por rango de capacidad (D18, `OCUPACION_POR_RANGO.parquet`):** para cada rango r, la demanda que necesita r **o más** contra los espacios de r **o más**. `PROB_SATURA_ACUM` es el % de simulaciones calibradas en que esa demanda supera esos espacios. Con r = 0-20 es la `PROB_SATURA` de la celda. En `OCUPACION_Y_DEFICIT` se agregan `PROB_SATURA_TAMANO` (la mayor entre los rangos) y `RANGO_CRITICO` (el rango donde ocurre; ante empate, el mayor).

In [42]:
activos = inventario.groupby(["COD_CAMPUS_EDIFICIO", "CLASE_MAPA"]).size().rename("ESPACIOS_ACTIVOS")
filas_o = []
for (T, esc), comb in sims_guardadas.items():
    cap = np.array([activos.get((c, k), 0) for c in CAMP for _ in DIAS for _ in range(NB) for k in CLASES], dtype=float)
    prob = (comb > cap[None, :]).mean(0)
    filas_o.append(pd.DataFrame({"PERIODO": T, "ESCENARIO": esc, "CELDA": np.arange(N_CELDAS), "PROB_SATURA": prob, "CAPACIDAD": cap}))
o = pd.concat(filas_o, ignore_index=True)
camp_i, rest = np.divmod(o["CELDA"].to_numpy(), N_SC); dia_i, rest2 = np.divmod(rest, NB * len(CLASES)); blo, cla_i = np.divmod(rest2, len(CLASES))
o["COD_CAMPUS_EDIFICIO"] = np.array(list(CAMP))[camp_i]; o["DIA"] = np.array(DIAS)[dia_i]; o["BLOQUE"] = blo.astype("int64"); o["CLASE"] = np.array(CLASES)[cla_i]
ocup = mapa.merge(o.drop(columns="CELDA"), on=["PERIODO", "ESCENARIO", "COD_CAMPUS_EDIFICIO", "DIA", "BLOQUE", "CLASE"], how="left")
ocup["TASA_USO_P50"] = ocup["OCUPACION_P50"] / ocup["CAPACIDAD"].replace(0, np.nan)
deficit = (programacion[programacion["DEFICIT"]].groupby(["PERIODO", "COD_CAMPUS_EDIFICIO", "DIA", "BLOQUE", "CLASE"])["PESO_SEMANAS"]
           .sum().rename("DEFICIT_SALAS").reset_index().assign(ESCENARIO="base"))
ocup = ocup.merge(deficit, on=["PERIODO", "ESCENARIO", "COD_CAMPUS_EDIFICIO", "DIA", "BLOQUE", "CLASE"], how="left").fillna({"DEFICIT_SALAS": 0})
# D18 · alerta anidada por rango de capacidad
filas_r = []
for (T, esc), comb in sims_guardadas.items():
    acum = PARTE_RANGO[(T, esc)][:, ::-1].cumsum(1)[:, ::-1]           # parte de la demanda de la celda que necesita rango ≥ r
    cap_acum = CAP_ACUM_S[T % 100]
    for r_i in range(NR):
        dem = comb * acum[None, :, r_i]
        filas_r.append(pd.DataFrame({"PERIODO": T, "ESCENARIO": esc, "CELDA": np.arange(N_CELDAS), "RANGO_DESDE": RANGOS[r_i], "R_I": r_i,
                                     "DEMANDA_ACUM_P10": np.percentile(dem, 10, 0), "DEMANDA_ACUM_P50": np.percentile(dem, 50, 0),
                                     "DEMANDA_ACUM_P90": np.percentile(dem, 90, 0), "ESPACIOS_ACUM": cap_acum[:, r_i],
                                     "PROB_SATURA_ACUM": (dem > cap_acum[None, :, r_i]).mean(0)}))
por_rango = pd.concat(filas_r, ignore_index=True)
por_rango = por_rango[por_rango["DEMANDA_ACUM_P90"] > 0]
camp_i, rest = np.divmod(por_rango["CELDA"].to_numpy(), N_SC); dia_i, rest2 = np.divmod(rest, NB * len(CLASES)); blo, cla_i = np.divmod(rest2, len(CLASES))
por_rango = por_rango.assign(COD_CAMPUS_EDIFICIO=np.array(list(CAMP))[camp_i], DIA=np.array(DIAS)[dia_i], BLOQUE=blo.astype("int64"),
                             CLASE=np.array(CLASES)[cla_i], H=lambda d: d["PERIODO"].map(horizonte), VALIDADO=lambda d: d["PERIODO"].map(validado))
crit = (por_rango.sort_values(["PROB_SATURA_ACUM", "R_I"]).drop_duplicates(["PERIODO", "ESCENARIO", "CELDA"], keep="last")
        [["PERIODO", "ESCENARIO", "COD_CAMPUS_EDIFICIO", "DIA", "BLOQUE", "CLASE", "PROB_SATURA_ACUM", "RANGO_DESDE"]]
        .rename(columns={"PROB_SATURA_ACUM": "PROB_SATURA_TAMANO", "RANGO_DESDE": "RANGO_CRITICO"}))
ocup = ocup.merge(crit, on=["PERIODO", "ESCENARIO", "COD_CAMPUS_EDIFICIO", "DIA", "BLOQUE", "CLASE"], how="left").fillna({"PROB_SATURA_TAMANO": 0})
por_rango.drop(columns=["CELDA", "R_I"]).to_parquet(BBDD / "OCUPACION_POR_RANGO.parquet", index=False)
_chk = por_rango[por_rango["R_I"] == 0].merge(o, on=["PERIODO", "ESCENARIO", "COD_CAMPUS_EDIFICIO", "DIA", "BLOQUE", "CLASE"])
print("control: rango 0-20 = PROB_SATURA de la celda →", bool(np.allclose(_chk["PROB_SATURA_ACUM"], _chk["PROB_SATURA"])))
print(ocup[ocup["ESCENARIO"] == "base"].groupby("PERIODO").agg(SATURA_10=("PROB_SATURA", lambda v: int((v >= 0.1).sum())),
      SATURA_TAMANO_10=("PROB_SATURA_TAMANO", lambda v: int((v >= 0.1).sum())), SATURA_25=("PROB_SATURA", lambda v: int((v >= 0.25).sum())),
      SATURA_TAMANO_25=("PROB_SATURA_TAMANO", lambda v: int((v >= 0.25).sum()))).to_string())
ocup.to_parquet(BBDD / "OCUPACION_Y_DEFICIT.parquet", index=False)
pico = (ocup[ocup["ESCENARIO"] == "base"].sort_values("TASA_USO_P50", ascending=False)
        .groupby(["PERIODO", "COD_CAMPUS_EDIFICIO", "CLASE"]).head(1)
        .pivot_table(index=["COD_CAMPUS_EDIFICIO", "CLASE"], columns="PERIODO", values="TASA_USO_P50"))
(pico * 100).round(0).iloc[:, :5]

control: rango 0-20 = PROB_SATURA de la celda → True
         SATURA_10  SATURA_TAMANO_10  SATURA_25  SATURA_TAMANO_25
PERIODO                                                          
202705           0                 0          0                 0
202710         127               181         48                93
202715           0                 0          0                 0
202720          78               125         17                67
202725           0                 0          0                 0
202805           0                 0          0                 0
202810         234               297         79               125
202815           0                 0          0                 0
202820         173               223         36                87
202825           0                 1          0                 0
202905           0                 0          0                 0
202910         295               366        127               179
202915           0     

PERIODO                          202705  202710  202715  202720  202725
COD_CAMPUS_EDIFICIO CLASE                                              
C05                 LABORATORIO    18.0    50.0    17.0    51.0    18.0
                    SALA           40.0    57.0    38.0    56.0    43.0
C06                 LABORATORIO     0.0    71.0     0.0    69.0     0.0
                    SALA            0.0    98.0     0.0    88.0     0.0
C04                 LABORATORIO    13.0    68.0    14.0    67.0    13.0
                    SALA           44.0    99.0    43.0    99.0    43.0
C02                 LABORATORIO     5.0    61.0     5.0    61.0     4.0
                    SALA           48.0    85.0    44.0    80.0    42.0
C03                 LABORATORIO    11.0    62.0     7.0    56.0    10.0
                    SALA           30.0    87.0    38.0    89.0    33.0
C07                 LABORATORIO     0.0    51.0     0.0    65.0     0.0
                    SALA            0.0    77.0     0.0    68.0     0.0
C01                 LABORATORIO     6.0    62.0     5.0    64.0     5.0
                    SALA           29.0    92.0    34.0    91.0    35.0

## 8 · Controles y metadatos

1. **Consistencia con el banco:** C1 en 202620, con estas funciones y el volumen **real** como oráculo, debe reproducir el MASE del banco (`VALIDACION_RESULTADOS.parquet`) dentro de ±0,02.
2. **Cuadratura:** el volumen proyectado del escenario base 2027 contra la base.
3. **Metadatos** del corte: fechas, w, β, percentiles del estándar real y avisos.

In [43]:
# 1 · Consistencia con el banco en 202620 (oráculo de volumen = inscritos reales de las secciones físicas de 202620)
T_c, s_c = 202620, 20
B_c = T_c - 100
clase_c = clasificar([p for p in periodos if B_c - 200 <= p <= B_c])
secs_c, filas_c = secciones_fisicas(B_c, clase_c)
secs_t, _ = secciones_fisicas(T_c, clasificar([p for p in periodos if T_c - 200 <= p <= T_c]))
secs_c["EST"] = secs_c[ESTRATO].astype(str).agg("|".join, axis=1)
secs_t["EST"] = secs_t[ESTRATO].astype(str).agg("|".join, axis=1)
vb_c = secs_c.groupby("EST")["VOL"].sum()
obj_c = (vb_c * (secs_t.groupby("EST")["VOL"].sum().reindex(vb_c.index) / vb_c) ** elasticidad(T_c)).dropna()
f_c = filas_c.sort_values("SEC").reset_index(drop=True); f_cel = celda_de(f_c.rename(columns={"CAMP": "COD_CAMPUS_EDIFICIO"}))
rng_c = np.random.default_rng(SEMILLA); sims_c = []
for _ in range(100):
    el = remuestrear(secs_c[secs_c["EST"].isin(obj_c.index)], obj_c, "VOL", rng_c)
    ini = np.searchsorted(f_c["SEC"].to_numpy(), el["SEC"].to_numpy(), "left"); fin = np.searchsorted(f_c["SEC"].to_numpy(), el["SEC"].to_numpy(), "right")
    largo = fin - ini; idx = np.repeat(ini, largo) + (np.arange(largo.sum()) - np.repeat(np.cumsum(largo) - largo, largo))
    sims_c.append(np.bincount(f_cel[idx], weights=f_c["PESO"].to_numpy()[idx], minlength=N_CELDAS))
naive_c, real_c = mapa_real(B_c), mapa_real(T_c)
w_c = float(val[(val["MODELO"] == "C1") & (val["ORIGEN"] == T_c)]["W_NAIVE"].iat[0])     # w de C1-oráculo en el banco
pred_c = w_c * naive_c + (1 - w_c) * np.mean(sims_c, axis=0)
wape = lambda r, p: np.abs(p - r)[(r > 0) | (p > 0)].sum() / r[(r > 0) | (p > 0)].sum()
mase_c = wape(real_c, pred_c) / wape(real_c, naive_c)
ref = val[(val["MODELO"] == "C1") & (val["ORIGEN"] == T_c) & (val["GRANO"] == "celda")]["MASE"].iat[0]
print(f"{'OK    ' if abs(mase_c - ref) <= 0.02 else 'ALERTA'} C1 en 202620 (celda): Modelos {mase_c:.3f} · banco {ref:.3f}")

# 2 · Cuadratura del volumen 2027, escenario base
print(vol[(vol["PERIODO"] // 100 == ANIO_CORTO) & (vol["ESCENARIO"] == "base")].groupby("PERIODO")[["INSCRITOS_BASE", "INSCRITOS_PROY"]].sum().round(0))

# 3 · Metadatos
meta = {"fecha_corte": str(date.today()), "modelo_adoptado": MODELO_ADOPTADO or "naive", "fuente_volumen_2027": FUENTE_VOLUMEN,
        "base_por_sufijo": BASE, "w_naive": W_NAIVE, "beta": BETA,
        "estandar_real": {str(s): ESTANDAR[s].reset_index().astype(str).to_dict("records") for s in SUFIJOS},
        "rangos_capacidad": RANGOS, "tolerancia_capacidad": {str(s): t for s, t in TOL.items()},
        "corto": ANIO_CORTO, "largo": ANIOS_LARGO, "h_validado_hasta": H_VALIDADO, "escenarios": ESCENARIOS,
        "modelo_largo_plazo": {"modelo": MODELO_LP, "w_naive": W_NAIVE_LP, "desde_h": ESCALAR_DESDE_H,
                               "nota": "C1E (naive escalado, D19, Diego 28-10) en 2028-2031; 2027 con C1"},
        "regla_por_horizonte_informativa": {f"{n}_H{h}": m for (n, h), m in MODELO_H.items()},
        "k_banda_por_horizonte": {f"{n}_H{h}": round(k, 3) for (n, h), k in K_H.items()},
        "ensanche_ariadna": {f"{n}_H{h}": round(f, 3) for (n, h), f in ENSANCHE_ARI.items()},
        "avisos": ["largo plazo con C1E (naive escalado, D19): corrige el sesgo de C1 (H+4 -7,6 % -> -2,7 %) y mejora el MASE desde H+3, pero en H+2 (2028) lo empeora en 0,011; a 1 año no se usa porque pierde contra C1-TEN",
                   f"largo plazo validado con pseudo-backtest hasta H+{H_VALIDADO} con muestra chica (2 pares de pregrado en H+4, todos con objetivo 2026); H+5 (2031) extrapolado; el pronóstico de Ariadna entra solo por su propio backtest (H+1 a H+3, agregado por nivel)",
                   "banda P10-P90 conformal calibrada a 1 año (H5): cobertura media 75 % en UG, no pasa la regla estricta (subcubre 202420/202520); 2028-2031 ensanchada por k_h (D17); bajo/alto 2027 = cuantiles de Euribia sumados por tipo",
                   "con volumen proyectado la ganancia sobre el naive es marginal (D15: C1-TEN 3/5 orígenes UG, MASE medio 1,011)",
                   "base AV 202625 con inscripción abierta",
                   "rango de capacidad (D18): alerta anidada hacia arriba con los 4 rangos de la planta; tolerancia = P90 de inscritos/capacidad de la base por clase; el reparto por rango es informativo en el banco (no pasa por la regla de adopción)"],
        "consistencia_banco_202620_celda": {"modelos": round(float(mase_c), 4), "banco": round(float(ref), 4)}}
(BBDD / "MODELO_METADATOS.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("metadatos escritos")

OK     C1 en 202620 (celda): Modelos 0.942 · banco 0.944
         INSCRITOS_BASE  INSCRITOS_PROY
PERIODO                                
202705            46481         46124.0
202710           329627        345663.0
202715            44367         43746.0
202720           309211        327739.0
202725            40489         40119.0
metadatos escritos
